# Open-Meteo Weather Forecast EDA

This notebook inspects the Open-Meteo Previous Runs weather feature source before it is allowed to influence price-model experiments.

The point is not to make weather look useful. The point is to answer data-quality questions first:

1. Which model / lead / variable groups are actually present?
2. Which groups survive the v1 DK1/DK2 basket coverage gate?
3. How much does `forecast_available_at_utc` mask at a normal day-ahead origin?
4. Do the DK1/DK2 area aggregates look plausible?
5. What weather columns would survive in a small price-plus-weather experiment frame?

<span style="color:#b00020"><b>Interpretation policy:</b> red callouts are judgement notes. They are deliberately separate from the tables and plots so we do not confuse interpretation with the mechanics of loading and grouping the data.</span>

## Data Artifact Choice

The notebook prefers the canonical Open-Meteo v1 artifacts. If they have not been built yet, it falls back to the focused EDA artifact created for this pass:

```bash
python scripts/fetch_open_meteo_previous_runs.py \
  --start 2025-01-01 --end 2025-01-14 \
  --models gfs_global icon_eu metno_nordic \
  --raw-dir data/raw/open_meteo_eda

python scripts/build_open_meteo_weather_features.py \
  --raw-dir data/raw/open_meteo_eda \
  --dataset-version open_meteo_previous_runs_v1_eda
```

The focused artifact is only an EDA sample. Use the canonical build for real model comparisons.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = Path("/Users/peterbjerrehansen/Desktop/projects/coding_projects/on_github/dk_energy_forecasts")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dkenergy_data.build.open_meteo_weather_v1 import OPEN_METEO_LOCATION_BASKET
from dkenergy_forecast.features.weather_features import build_weather_experiment_frame, weather_value_columns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)


COPENHAGEN_TZ = "Europe/Copenhagen"

In [ ]:
ARTIFACT_CANDIDATES = [
    {
        "label": "canonical_v1",
        "normalized": PROJECT_ROOT / "data" / "normalized" / "open_meteo_previous_runs_open_meteo_previous_runs_v1.parquet",
        "area_long": PROJECT_ROOT / "data" / "features" / "weather_open_meteo_area_hourly_long_open_meteo_previous_runs_v1.parquet",
        "area_wide": PROJECT_ROOT / "data" / "features" / "weather_open_meteo_area_hourly_wide_open_meteo_previous_runs_v1.parquet",
        "qa": PROJECT_ROOT / "data" / "features" / "weather_open_meteo_area_hourly_open_meteo_previous_runs_v1.qa.json",
    },
    {
        "label": "focused_eda_2025_01_01_to_2025_01_14",
        "normalized": PROJECT_ROOT / "data" / "normalized" / "open_meteo_previous_runs_open_meteo_previous_runs_v1_eda.parquet",
        "area_long": PROJECT_ROOT / "data" / "features" / "weather_open_meteo_area_hourly_long_open_meteo_previous_runs_v1_eda.parquet",
        "area_wide": PROJECT_ROOT / "data" / "features" / "weather_open_meteo_area_hourly_wide_open_meteo_previous_runs_v1_eda.parquet",
        "qa": PROJECT_ROOT / "data" / "features" / "weather_open_meteo_area_hourly_open_meteo_previous_runs_v1_eda.qa.json",
    },
]

artifact = next(
    (candidate for candidate in ARTIFACT_CANDIDATES if candidate["normalized"].exists() and candidate["area_long"].exists()),
    None,
)
if artifact is None:
    raise FileNotFoundError(
        "No Open-Meteo weather artifacts found. Run the fetch/build commands in the previous cell first."
    )

normalized = pd.read_parquet(artifact["normalized"])
area_long = pd.read_parquet(artifact["area_long"])
area_wide = pd.read_parquet(artifact["area_wide"]) if artifact["area_wide"].exists() else pd.DataFrame()
qa = json.loads(artifact["qa"].read_text(encoding="utf-8")) if artifact["qa"].exists() else {}

for frame, time_columns in [
    (normalized, ["valid_time_utc", "forecast_available_at_utc"]),
    (area_long, ["ds_utc", "forecast_available_at_utc"]),
    (area_wide, ["ds_utc"]),
]:
    for column in time_columns:
        if column in frame.columns:
            frame[column] = pd.to_datetime(frame[column], utc=True)

normalized["lead_label"] = "lead" + normalized["lead_time_days"].astype(str) + "d"
area_long["lead_label"] = "lead" + area_long["lead_time_days"].astype(str) + "d"
area_long["model_lead"] = area_long["weather_model"] + " / " + area_long["lead_label"]
area_long["local_time"] = area_long["ds_utc"].dt.tz_convert(COPENHAGEN_TZ)
area_long["local_date"] = area_long["local_time"].dt.strftime("%Y-%m-%d")
area_long["local_hour"] = area_long["local_time"].dt.hour

summary = {
    "artifact_label": artifact["label"],
    "normalized_rows": int(len(normalized)),
    "area_feature_rows": int(len(area_long)),
    "weather_models": sorted(normalized["weather_model"].dropna().unique().tolist()),
    "lead_time_days": sorted(int(value) for value in normalized["lead_time_days"].dropna().unique()),
    "parameters": sorted(normalized["parameter_id"].dropna().unique().tolist()),
    "valid_time_min": normalized["valid_time_utc"].min().isoformat(),
    "valid_time_max": normalized["valid_time_utc"].max().isoformat(),
    "qa_feature_rows_usable": qa.get("feature_rows_usable_count"),
    "qa_feature_groups_passing": qa.get("feature_groups_passing_count"),
    "qa_feature_groups_total": qa.get("feature_groups_total_count"),
}
summary

<span style="color:#b00020"><b>Interpretation:</b> the focused EDA artifact is intentionally small. Treat it as a source-shape and coverage diagnostic, not as evidence about annual weather-price relationships.</span>

## Coordinate Basket Sanity

The v1 weather manifest uses five point forecasts for DK1 and five for DK2. Aggregation is a simple unweighted mean across available points, with no interpolation.

In [ ]:
location_manifest = pd.DataFrame([location.__dict__ for location in OPEN_METEO_LOCATION_BASKET])
location_manifest

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for area, group in location_manifest.groupby("area"):
    ax.scatter(group["longitude"], group["latitude"], s=80, label=area)
    for _, row in group.iterrows():
        ax.annotate(row["location_id"].replace("dk1_", "").replace("dk2_", ""), (row["longitude"], row["latitude"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set_title("Open-Meteo v1 coordinate basket")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(title="Area")
fig.tight_layout()
plt.show()

<span style="color:#b00020"><b>Interpretation:</b> the basket is not a geographic polygon approximation. It is a pragmatic first pass: enough spread to avoid a single-city proxy, simple enough that coverage and provenance remain transparent.</span>

## Raw Variable Completeness

This check happens before area aggregation. It answers: did Open-Meteo return values for the requested variable/model/lead at the point level?

In [ ]:
completeness = (
    normalized.groupby(["weather_model", "lead_time_days", "parameter_id"], as_index=False)
    .agg(rows=("value", "size"), nonnull=("value", "count"))
)
completeness["nonnull_ratio"] = completeness["nonnull"] / completeness["rows"]
completeness = completeness.sort_values(["weather_model", "lead_time_days", "parameter_id"]).reset_index(drop=True)
completeness

In [ ]:
completeness_pivot = completeness.pivot_table(
    index=["weather_model", "lead_time_days"],
    columns="parameter_id",
    values="nonnull_ratio",
)
fig, ax = plt.subplots(figsize=(12, 4.8))
sns.heatmap(
    completeness_pivot,
    vmin=0,
    vmax=1,
    cmap="viridis",
    annot=True,
    fmt=".0%",
    linewidths=0.5,
    cbar_kws={"label": "Point-level non-null ratio"},
    ax=ax,
)
ax.set_title("Open-Meteo point-level completeness by model, lead, and variable")
ax.set_xlabel("")
ax.set_ylabel("Model / lead")
fig.tight_layout()
plt.show()

<span style="color:#b00020"><b>Interpretation:</b> in the focused January sample, ICON-EU is complete for both requested leads, MET Norway Nordic is usable for lead 1 but not lead 2, and GFS has uneven previous-run availability. This justifies model-specific coverage gates and a common-overlap table later.</span>

## Area Feature Coverage Gates

The v1 feature gate has two layers:

1. `location_coverage_pass`: the individual area-hour has enough basket points.
2. `feature_group_pass`: the full area/model/lead/parameter/window has enough usable hours.

The joined experiment frame requires both by default.

In [ ]:
feature_groups = area_long[
    [
        "area",
        "weather_model",
        "lead_time_days",
        "parameter_id",
        "feature_window_coverage_ratio",
        "feature_group_pass",
    ]
].drop_duplicates()

coverage_summary = (
    feature_groups.groupby(["weather_model", "lead_time_days", "parameter_id"], as_index=False)
    .agg(
        area_count=("area", "nunique"),
        mean_window_coverage=("feature_window_coverage_ratio", "mean"),
        min_window_coverage=("feature_window_coverage_ratio", "min"),
        area_pass_rate=("feature_group_pass", "mean"),
    )
    .sort_values(["weather_model", "lead_time_days", "parameter_id"])
    .reset_index(drop=True)
)
coverage_summary

In [ ]:
coverage_pivot = coverage_summary.pivot_table(
    index=["weather_model", "lead_time_days"],
    columns="parameter_id",
    values="area_pass_rate",
)
fig, ax = plt.subplots(figsize=(12, 4.8))
sns.heatmap(
    coverage_pivot,
    vmin=0,
    vmax=1,
    cmap="mako",
    annot=True,
    fmt=".0%",
    linewidths=0.5,
    cbar_kws={"label": "Share of areas passing v1 coverage gate"},
    ax=ax,
)
ax.set_title("Area feature groups passing the v1 coverage gate")
ax.set_xlabel("")
ax.set_ylabel("Model / lead")
fig.tight_layout()
plt.show()

In [ ]:
coverage_by_area = (
    feature_groups.groupby(["area", "weather_model", "lead_time_days"], as_index=False)
    .agg(feature_groups=("parameter_id", "count"), passing_groups=("feature_group_pass", "sum"))
)
coverage_by_area["passing_share"] = coverage_by_area["passing_groups"] / coverage_by_area["feature_groups"]

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(data=coverage_by_area, x="weather_model", y="passing_share", hue="lead_time_days", ax=ax)
ax.set_title("Coverage-gated feature share by model and lead")
ax.set_xlabel("")
ax.set_ylabel("Passing feature share")
ax.set_ylim(0, 1.05)
ax.legend(title="Lead days")
fig.tight_layout()
plt.show()
coverage_by_area

<span style="color:#b00020"><b>Interpretation:</b> the coverage gate is doing real work. Without it, the feature table would contain columns that look official but are mostly null or irregularly present. That would make backtest comparisons noisy and potentially misleading.</span>

## Forecast-Origin Availability Masking

Coverage is not the same as availability. `previous_day1` can be present in the source table but unavailable for late delivery hours at a 10:00 UTC day-ahead origin.

The next cell asks: for one sample origin, how many delivery-day hours survive the availability rule?

In [ ]:
def delivery_day_rows(area_features: pd.DataFrame, origin: pd.Timestamp) -> pd.DataFrame:
    origin = pd.Timestamp(origin).tz_convert("UTC") if pd.Timestamp(origin).tzinfo else pd.Timestamp(origin).tz_localize("UTC")
    delivery_local_date = (origin.tz_convert(COPENHAGEN_TZ) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    rows = area_features[area_features["local_date"] == delivery_local_date].copy()
    rows["sample_forecast_origin_utc"] = origin
    rows["available_at_origin"] = rows["forecast_available_at_utc"] <= origin
    rows["usable_at_origin"] = rows["available_at_origin"] & rows["feature_group_pass"] & rows["location_coverage_pass"]
    return rows

sample_origin = pd.Timestamp("2025-01-05T10:00:00Z")
delivery = delivery_day_rows(area_long, sample_origin)

availability_summary = (
    delivery.groupby(["weather_model", "lead_time_days", "parameter_id"], as_index=False)
    .agg(
        delivery_hours=("ds_utc", "nunique"),
        rows=("ds_utc", "size"),
        rows_available=("available_at_origin", "sum"),
        rows_usable=("usable_at_origin", "sum"),
    )
)
availability_summary["available_ratio"] = availability_summary["rows_available"] / availability_summary["rows"]
availability_summary["usable_ratio"] = availability_summary["rows_usable"] / availability_summary["rows"]
availability_summary.sort_values(["weather_model", "lead_time_days", "parameter_id"])

In [ ]:
availability_pivot = availability_summary.pivot_table(
    index=["weather_model", "lead_time_days"],
    columns="parameter_id",
    values="usable_ratio",
)
fig, ax = plt.subplots(figsize=(12, 4.8))
sns.heatmap(
    availability_pivot,
    vmin=0,
    vmax=1,
    cmap="rocket_r",
    annot=True,
    fmt=".0%",
    linewidths=0.5,
    cbar_kws={"label": "Rows usable at sample forecast origin"},
    ax=ax,
)
ax.set_title(f"Weather rows usable at {sample_origin.isoformat()}")
ax.set_xlabel("")
ax.set_ylabel("Model / lead")
fig.tight_layout()
plt.show()

In [ ]:
mask_example = delivery[
    (delivery["area"] == "DK1")
    & (delivery["weather_model"] == "icon_eu")
    & (delivery["parameter_id"] == "temperature_2m")
    & (delivery["feature_group_pass"])
    & (delivery["location_coverage_pass"])
].copy()
mask_example["lead_label"] = "lead" + mask_example["lead_time_days"].astype(str) + "d"

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.scatterplot(
    data=mask_example,
    x="local_hour",
    y="value",
    hue="lead_label",
    style="available_at_origin",
    s=90,
    ax=ax,
)
ax.set_title("ICON-EU DK1 temperature: available vs masked at sample origin")
ax.set_xlabel("Delivery local hour")
ax.set_ylabel("Temperature forecast, degC")
ax.legend(title="Lead / available")
fig.tight_layout()
plt.show()

<span style="color:#b00020"><b>Interpretation:</b> availability masking is not a theoretical nicety. For a 10:00 UTC origin, lead 1 tends to be useful for early delivery hours and masked for later delivery hours; lead 2 is often what keeps the full delivery day weather-covered.</span>

## DK1/DK2 Area Behavior

This is a plausibility check on the unweighted area aggregation. We are looking for smooth, sensible weather signals and obvious DK1/DK2 differences, not trying to prove predictive value yet.

In [ ]:
def select_area_series(model="icon_eu", lead=1, parameters=("temperature_2m", "wind_speed_10m", "shortwave_radiation")):
    return area_long[
        (area_long["weather_model"] == model)
        & (area_long["lead_time_days"] == lead)
        & (area_long["parameter_id"].isin(parameters))
        & (area_long["feature_group_pass"])
        & (area_long["location_coverage_pass"])
    ].copy()

area_series = select_area_series()

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
for ax, parameter in zip(axes, ["temperature_2m", "wind_speed_10m", "shortwave_radiation"]):
    plot_data = area_series[area_series["parameter_id"] == parameter]
    sns.lineplot(data=plot_data, x="ds_utc", y="value", hue="area", ax=ax)
    ax.set_title(parameter)
    ax.set_xlabel("")
    ax.set_ylabel("Forecast value")
    ax.legend(title="Area")
fig.suptitle("ICON-EU lead 1 area-hour weather features", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
area_wide_small = area_series.pivot_table(
    index=["ds_utc", "parameter_id"],
    columns="area",
    values="value",
    aggfunc="last",
).dropna().reset_index()
area_wide_small["dk1_minus_dk2"] = area_wide_small["DK1"] - area_wide_small["DK2"]
area_difference_summary = (
    area_wide_small.groupby("parameter_id", as_index=False)
    .agg(
        observations=("dk1_minus_dk2", "size"),
        mean_dk1_minus_dk2=("dk1_minus_dk2", "mean"),
        median_dk1_minus_dk2=("dk1_minus_dk2", "median"),
        p10_dk1_minus_dk2=("dk1_minus_dk2", lambda s: s.quantile(0.10)),
        p90_dk1_minus_dk2=("dk1_minus_dk2", lambda s: s.quantile(0.90)),
    )
)
area_difference_summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(data=area_wide_small, x="parameter_id", y="dk1_minus_dk2", ax=ax)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("DK1 minus DK2 area-feature differences, ICON-EU lead 1")
ax.set_xlabel("")
ax.set_ylabel("DK1 - DK2")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
plt.show()

<span style="color:#b00020"><b>Interpretation:</b> for this winter slice, temperature and wind move together across DK1 and DK2, but the differences are not zero. That is good enough to keep area-specific weather features instead of collapsing Denmark to one national weather vector.</span>

## Model-to-Model Comparison

Here we compare model forecasts only where both model groups pass the v1 coverage gate. The goal is to see whether model-specific features carry distinct information or merely duplicate each other.

In [ ]:
comparison_params = ["temperature_2m", "wind_speed_10m", "cloud_cover"]
comparison = area_long[
    (area_long["lead_time_days"] == 1)
    & (area_long["parameter_id"].isin(comparison_params))
    & (area_long["feature_group_pass"])
    & (area_long["location_coverage_pass"])
].copy()
model_wide = comparison.pivot_table(
    index=["area", "ds_utc", "parameter_id"],
    columns="weather_model",
    values="value",
    aggfunc="last",
).reset_index()

pairs = []
for left, right in [("icon_eu", "metno_nordic"), ("icon_eu", "gfs_global"), ("metno_nordic", "gfs_global")]:
    if {left, right}.issubset(model_wide.columns):
        pair = model_wide.dropna(subset=[left, right]).copy()
        if not pair.empty:
            pair["left_model"] = left
            pair["right_model"] = right
            pair["difference"] = pair[left] - pair[right]
            pair["abs_difference"] = pair["difference"].abs()
            pairs.append(pair[["area", "ds_utc", "parameter_id", "left_model", "right_model", "difference", "abs_difference"]])
model_differences = pd.concat(pairs, ignore_index=True) if pairs else pd.DataFrame()

model_difference_summary = (
    model_differences.groupby(["left_model", "right_model", "parameter_id"], as_index=False)
    .agg(
        rows=("abs_difference", "size"),
        mean_abs_difference=("abs_difference", "mean"),
        p90_abs_difference=("abs_difference", lambda s: s.quantile(0.90)),
    )
    if not model_differences.empty
    else pd.DataFrame()
)
model_difference_summary

In [ ]:
if not model_differences.empty:
    plot_data = model_differences[model_differences["left_model"].eq("icon_eu") & model_differences["right_model"].eq("metno_nordic")]
    fig, ax = plt.subplots(figsize=(9, 4.5))
    sns.boxplot(data=plot_data, x="parameter_id", y="difference", hue="area", ax=ax)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title("ICON-EU minus MET Norway Nordic, lead 1")
    ax.set_xlabel("")
    ax.set_ylabel("Forecast difference")
    ax.tick_params(axis="x", rotation=20)
    fig.tight_layout()
    plt.show()
else:
    print("No overlapping model pairs available after coverage gating.")

<span style="color:#b00020"><b>Interpretation:</b> ICON-EU and MET Norway Nordic are not identical in this sample. Keeping model-specific feature groups for Phase 2 is therefore reasonable; the ensemble summaries should be an addition, not a replacement.</span>

## Price-Panel Join Smoke

This section does not train a model. It creates a tiny availability-masked experiment frame so we can inspect which weather columns would survive alongside the existing price/calendar/lag features.

In [ ]:
price_candidates = [
    PROJECT_ROOT / "data" / "model_ready" / "price_panel_hourly_v1.parquet",
    PROJECT_ROOT / "data" / "model_ready" / "price_panel_hourly_v1_eda.parquet",
]
price_path = next((path for path in price_candidates if path.exists()), None)

if price_path is None:
    print("No price panel found; skipping experiment-frame smoke.")
    experiment = pd.DataFrame()
else:
    panel = pd.read_parquet(price_path)
    panel["ds_utc"] = pd.to_datetime(panel["ds_utc"], utc=True)
    # Pick origins whose delivery days are inside the focused weather sample.
    origins = pd.DataFrame(
        {
            "forecast_origin_utc": pd.to_datetime(
                ["2025-01-04T10:00:00Z", "2025-01-05T10:00:00Z", "2025-01-06T10:00:00Z"],
                utc=True,
            )
        }
    )
    experiment = build_weather_experiment_frame(panel, origins, area_long)
    print(f"Experiment rows: {len(experiment):,}")
    print(f"Experiment columns: {len(experiment.columns):,}")
    print(f"Price panel: {price_path.relative_to(PROJECT_ROOT)}")

In [ ]:
if experiment.empty:
    weather_column_summary = pd.DataFrame()
else:
    value_columns = weather_value_columns(experiment)
    weather_column_summary = pd.DataFrame(
        {
            "feature": value_columns,
            "nonnull_ratio": [experiment[column].notna().mean() for column in value_columns],
            "nonnull_rows": [int(experiment[column].notna().sum()) for column in value_columns],
        }
    ).sort_values(["nonnull_ratio", "feature"], ascending=[False, True]).reset_index(drop=True)
weather_column_summary.head(30)

In [ ]:
if not weather_column_summary.empty:
    plot_data = weather_column_summary.head(35).copy()
    fig, ax = plt.subplots(figsize=(10, max(6, len(plot_data) * 0.23)))
    sns.barplot(data=plot_data, y="feature", x="nonnull_ratio", ax=ax, color="#3478a6")
    ax.set_title("Top availability-masked weather feature columns in smoke frame")
    ax.set_xlabel("Non-null share")
    ax.set_ylabel("")
    ax.set_xlim(0, 1.05)
    fig.tight_layout()
    plt.show()

<span style="color:#b00020"><b>Interpretation:</b> this smoke frame is the bridge from data quality to modeling. If a model group has no non-null weather columns after coverage and availability masking, the ablation runner must skip it. Otherwise we would accidentally compare the baseline against itself under a weather label.</span>

## EDA Takeaways

<span style="color:#b00020"><b>Interpretation:</b></span>

1. ICON-EU looks like the cleanest MVP backbone in the focused sample.
2. MET Norway Nordic should be treated as mostly lead-1 for this source/version unless fuller backfills prove otherwise.
3. GFS should stay in the experiment set, but only through coverage-gated columns. Several requested `previous_day` fields are sparse or missing in the sample.
4. Lead-specific availability masking is mandatory. `previous_day1` is not automatically safe for an entire Danish delivery day at a 10:00 UTC origin.
5. DK1/DK2 area features are close but not identical, so keeping area-specific weather is preferable to a single national weather proxy.
6. The next serious step is a full or deliberately bounded Open-Meteo backfill, then weather ablations with both model-specific max-history results and common-overlap results.